# Fine-tuning de un SLM para cumplimiento regulatorio CNBV / Banxico

Elaborado para el **AWS Summit Ciudad de México 2026**

Presentado por: Arturo Minor (Sr. SA AWS), Oscar Ramírez (Sr. AI Specialist SA) y David Sánchez (Sr. SA)

### Introducción

Este notebook detalla, de principio a fin, el proceso de continued training (fine-tuning) de
un **Small Language Model (SLM)** especializado en el dominio de **cumplimiento regulatorio
mexicano** (CNBV y Banxico) para la industria de servicios financieros, esta disenado para dar soporte a agentes que evaluan carpetas de
cumplimiento y estructuran documentos regulatorios a **baja latencia y bajo costo**.

El flujo completo es:

1. **Infraestructura** (AWS CDK): buckets S3 y roles IAM.
2. **Adquisicion de datos**: scraping de normatividad de CNBV y Banxico.
3. **Extraccion de texto** de los PDFs descargados.
4. **Chunking** de los documentos.
5. **Construccion del dataset** de instruccion (SFT) asistida por un LLM.
6. **Seleccion de modelo base y técnica de fine-tuning**, y catálogo de modelos candidatos
   validado contra `docs/technical_documentation.md`.
7. **Fine-tuning** con QLoRA en uno o varios SageMaker Training Jobs **en paralelo**.
8. **Analisis de métricas** del entrenamiento (tiempo, memoria, loss) y **evaluación de
   rendimiento** (perplexity, tokens/s, latencia) de cada modelo resultante, visualizadas con
   `pandas`/`seaborn`.
9. **Despliegue de prueba** sobre Bedrock AgentCore Runtime.
10. **Portabilidad** del modelo a la región `mx-central-1`.

> **Región de trabajo:** `us-west-2`. Es donde AgentCore Runtime esta soportado y donde la
> cuenta dispone de cuotas de GPU (`ml.g6.*`). `mx-central-1` (requisito de negocio final)
> no soporta AgentCore Runtime ni Custom Model Import, y no ofrece GPU; la sección 10
> explica como se logra la portabilidad hacia esa región.

> **Nota de ejecucion:** este notebook esta disenado para ejecutarse de principio a fin
> dentro de **Amazon SageMaker Studio**. Las celdas de código reproducen los comandos y el
> código reales del repositorio. Las etapas que consumen servicios de AWS (scraping masivo,
> Bedrock, SageMaker, AgentCore) estan pensadas para ejecutarse de forma controlada desde la
> Terminal integrada de Studio y se marcan como tales; las celdas de **analisis** (lectura de
> métricas y del dataset ya generado) no llaman a AWS y corren directamente en esta sesión.

## Índice

- [Objetivo y especificacion](#Objetivo-y-especificación)
- [Arquitectura del pipeline](#Arquitectura-del-pipeline)
- [0. Como ejecutar este notebook](#0.-Cómo-ejecutar-este-notebook)
- [1. Infraestructura (AWS CDK)](#1.-Infraestructura-(AWS-CDK))
  - [1.1 Diseno del datalake para el ciclo continuo de fine-tuning](#1.1-Diseño-del-datalake-para-el-ciclo-continuo-de-fine-tuning)
- [2. Adquisicion de datos: scraping de CNBV y Banxico](#2.-Adquisición-de-datos:-scraping-de-CNBV-y-Banxico)
  - [2.1 CNBV](#2.1-CNBV:-`data_pipeline/scraping/scrape_cnbv.py`)
  - [2.2 Banxico](#2.2-Banxico:-`data_pipeline/scraping/scrape_banxico.py`)
  - [2.3 Certificados TLS](#2.3-Detalle-técnico:-certificados-TLS-(`ca_bundle.py`))
- [3. Extraccion de texto](#3.-Extracción-de-texto:-`processing/extract_text.py`)
- [4. Chunking](#4.-Chunking:-`processing/chunk_documents.py`)
- [5. Construccion del dataset de instruccion (SFT)](#5.-Construcción-del-dataset-de-instrucción-(SFT):-`processing/build_dataset.py`)
  - [5.1 Inspeccion del dataset generado](#5.1-Inspección-del-dataset-generado)
- [6. Seleccion del modelo base y de la técnica de fine-tuning](#6.-Selección-del-modelo-base-y-de-la-técnica-de-fine-tuning)
  - [6.1 Modelo base: Qwen2.5-1.5B-Instruct](#6.1-Modelo-base:-Qwen2.5-1.5B-Instruct)
  - [6.2 Técnica: QLoRA](#6.2-Técnica:-QLoRA)
  - [6.3 Catálogo de modelos para fine-tuning en paralelo](#6.3-Catálogo-de-modelos-para-fine-tuning-en-paralelo-(validación-contra-`docs/technical_documentation.md`))
- [7. Fine-tuning con QLoRA en SageMaker](#7.-Fine-tuning-con-QLoRA-en-SageMaker)
  - [7.1 Hiperparametros del job](#7.1-Hiperparámetros-del-job)
  - [7.2 Configuracion de cuantización 4-bit](#7.2-Configuración-de-cuantización-4-bit-(dentro-de-`train_qlora.py`))
  - [7.3 Lanzamiento del training job](#7.3-Lanzamiento-del-training-job)
  - [7.4 Fine-tuning en paralelo de múltiples modelos candidatos](#7.4-Fine-tuning-en-paralelo-de-múltiples-modelos-candidatos)
- [8. Analisis de métricas del entrenamiento](#8.-Análisis-de-métricas-del-entrenamiento)
  - [8.1 Corrida individual (legacy)](#8.1-Corrida-individual-(legacy))
  - [8.2 Comparativo multi-modelo en paralelo](#8.2-Comparativo-multi-modelo-en-paralelo-(tiempo,-memoria,-loss))
  - [8.3 Evaluación de rendimiento post-entrenamiento](#8.3-Evaluación-de-rendimiento-post-entrenamiento-(calidad-y-velocidad))
- [9. Despliegue de prueba en Bedrock AgentCore Runtime](#9.-Despliegue-de-prueba-en-Bedrock-AgentCore-Runtime)
  - [9.1 El agente](#9.1-El-agente:-`agent_runtime/app.py`)
  - [9.2 La imagen](#9.2-La-imagen:-`agent_runtime/Dockerfile`)
  - [9.3 El stack](#9.3-El-stack:-`AgentRuntimeStack`)
- [10. Portabilidad a `mx-central-1`](#10.-Portabilidad-a-`mx-central-1`-(Intel-/-ARM,-sin-GPU))
- [Resumen y conclusiones](#Resumen-y-conclusiones)

## Objetivo y especificación

La presente solución de IA generativa que permite entrenar un Small Language Model (SLM) en `us-west-2` con
SageMaker, para luego realizar **fine-tuning con datos públicos del dominio de cumplimiento
regulatorio de la CNBV y Banxico**. El modelo resultante debe:

- Soportar casos de uso donde uno o varios **agentes** relacionados entre si bajo una arquitectura y un framework multi-agente usen el SLM para **evaluar carpetas** de cumplimiento.
  de la CNBV/Banxico y **estructurar documentos** de cumplimiento.
- Operar a **baja latencia y bajo costo**.
- Poder desplegarse en la región de AWS de `mx-central-1`, región que **solo cuenta con cómputo Intel (x86) y ARM (Graviton)**, sin GPU por el momento.

El proceso, por tanto, exige:

1. Identificar los datos de valor,
2. prepararlos,
3. elegir la técnica de fine-tuning más adecuada al modelo y a las restricciones de despliegue,
4. entrenar capturando métricas (duración, costo, loss), y finalmente,
5. montar el modelo sobre un runtime
de Bedrock AgentCore como prueba.


### Aviso sobre el uso de datos públicos de CNBV y Banxico

Los documentos normativos (circulares, disposiciones de carácter general, leyes, acuerdos) que
se descargan en la sección 2 son **información pública** que la Comisión Nacional Bancaria y de
Valores (CNBV) y el Banco de México publican en sus portales oficiales
([cnbv.gob.mx](https://www.cnbv.gob.mx/paginas/normatividad.aspx) y
[banxico.org.mx](https://www.banxico.org.mx)) en cumplimiento de sus obligaciones de
transparencia bajo la Ley General de Transparencia y Acceso a la Información Pública. Esto
significa:

- **No se usan datos personales, confidenciales ni reservados.** El corpus consiste
  exclusivamente en el texto de disposiciones normativas ya publicadas en el Diario Oficial de
  la Federación o en los portales institucionales; no hay expedientes de supervisión,
  información de instituciones financieras individuales, ni datos de personas físicas.
- **Este pipeline es un ejercicio técnico/demostrativo** para el AWS Summit, no un producto
  regulado. El SLM resultante **no sustituye asesoría legal ni la consulta directa de la fuente
  oficial**: la normatividad puede ser modificada, abrogada o interpretada por la propia CNBV o
  Banxico, y solo el texto publicado en el DOF tiene validez jurídica.
- **Atribución de la fuente.** Cada documento descargado conserva su metadata de origen
  (`metadata.jsonl`: URL, sector, fecha de publicación en el DOF) precisamente para poder citar
  la fuente original en cualquier respuesta que el modelo genere, tal como indica el
  `SYSTEM_PROMPT_SLM` de la sección 5.
- **Buenas prácticas de scraping.** Los scrapers (`scrape_cnbv.py`, `scrape_banxico.py`) hacen
  peticiones espaciadas y de un único hilo para no saturar los servidores de origen; no se
  eluden controles de acceso ni se descarga contenido restringido.

> Este notebook no constituye una interpretación legal de la Ley General de Transparencia y
> Acceso a la Información Pública; se referencia únicamente para contextualizar por qué esta
> normatividad es de acceso público.

## Arquitectura del pipeline

<!-- TODO: Add diagram -->
<!-- Fuente editable: docs/diagrams/01_arquitectura_pipeline.drawio -->

El repositorio se organiza así:

| Carpeta | Rol |
|---|---|
| `infra/` | CDK (Python): `DataPipelineStack`, `TrainingStack`, `AgentRuntimeStack` |
| `data_pipeline/scraping/` | Scrapers de CNBV y Banxico, y utilidad de certificados TLS |
| `data_pipeline/processing/` | Extracción de texto, chunking y generación del dataset |
| `training/` | Lanzador del training job y entry point QLoRA |
| `agent_runtime/` | Agente servido en AgentCore Runtime (imagen de contenedor) |
| `docs/` | Decisiones de arquitectura y guía de portabilidad |


## Guía rápida: dónde y cómo se ejecuta cada paso

Este notebook está pensado para ejecutarse **completo dentro de Amazon SageMaker Studio**:
tanto las celdas del notebook (análisis) como los comandos de reproducción (scraping,
Bedrock, SageMaker Training, despliegue de CDK, AgentCore) corren desde el mismo entorno,
usando la **Terminal integrada de Studio** (menú File → New → Terminal) para los segundos.

### Mapa de ejecución por sección

| Sección | Qué hace | Dónde se ejecuta | ¿Llama a AWS? | Tiempo aprox. |
|---|---|---|---|---|
| 0 | Verifica dependencias del notebook | Celda del notebook | No | 5 seg |
| 0.1 | Lee bucket/rol de entrenamiento desde Secrets Manager | Celda del notebook (boto3) | Sí (Secrets Manager, con fallback a CloudFormation) | 5 seg |
| 1 | Despliega infraestructura (CDK) | Terminal de Studio | Sí (CloudFormation) | 3-5 min |
| 2 | Scraping de CNBV y Banxico | Terminal de Studio | Sí (S3 sync) | 15-25 min |
| 3 | Extracción de texto de PDFs | Terminal de Studio | No (procesa archivos del entorno) | 2-5 min |
| 4 | Chunking de documentos | Terminal de Studio | No (procesa archivos del entorno) | 1-2 min |
| 5 | Generación del dataset (Bedrock) | Terminal de Studio | Sí (Bedrock) | 30-60 min |
| 5.1 | Inspección del dataset | Celda del notebook | No | 5 seg |
| 6 | Explicación de modelo/técnica | Solo lectura (markdown) | No | — |
| 7.3 / 7.4 | Lanza el/los training job(s) | Terminal de Studio | Sí (SageMaker) | 60-100 min |
| 8 | Análisis de métricas | Celda del notebook | No | 5 seg |
| 9 | Despliegue en AgentCore | Terminal de Studio | Sí (AgentCore) | 10-15 min |
| 10 | Portabilidad a mx-central-1 | Solo lectura (markdown) | No | — |

### Prerrequisitos y permisos IAM

El **rol de ejecución de SageMaker Studio** (el rol con el que arranca tu notebook/espacio)
necesita los siguientes permisos además de los que ya trae por defecto:

| Política | Para qué |
|---|---|
| `S3DataLakeAccess` | Leer/escribir en el bucket del pipeline (scraping, dataset, modelo) |
| `BedrockInvokeAccess` | Invocar Claude Haiku 4.5 para generar el dataset (sección 5) |
| `PassTrainingRole` | Pasar el rol de entrenamiento a SageMaker al lanzar el job (sección 7) |
| `NotebookConfigSecretReadAccess` | Leer el secreto `slm/notebook-config` de Secrets Manager (bucket, rol, cuenta, región — sección 0.1) |
| `CloudFormationReadAccess` | Respaldo: leer los outputs de los stacks de CDK si el secreto de Secrets Manager aún no existe (sección 0.1) |
| Permisos de CDK (opcional) | Si despliegas la infraestructura (sección 1) desde la propia Terminal de Studio, el rol también necesita los permisos de CloudFormation/IAM/S3 que requiere un `cdk deploy` |

Estos permisos se otorgan **una sola vez**, con una identidad que ya tenga privilegios de
administrador (por ejemplo, AWS CloudShell o la CLI de un administrador; un rol no puede
concederse permisos a sí mismo). Sustituye `<STUDIO_ROLE_NAME>` por el nombre de tu rol de
ejecución de Studio (lo encuentras en SageMaker Studio → User Details → Execution Role):

In [ ]:
EXECUTION_ROLE = "SAGEMAKER_EXECUTION_ROLE"
DATA_BUCKET = "DATA_BUCKET"

In [ ]:
import boto3
import json

# Conexión y obtención de llaves de AWS Secrets Manager
client = boto3.client('secretsmanager')
execution_role = json.loads(boto3.client('secretsmanager').get_secret_value(SecretId='slm/sagemaker')['SecretString'])[EXECUTION_ROLE]
data_bucket = json.loads(boto3.client('secretsmanager').get_secret_value(SecretId='slm/sagemaker')['SecretString'])[DATA_BUCKET]

### Prerrequisitos y permisos IAM

El **rol de ejecución de SageMaker Studio** (el rol con el que arranca tu notebook/espacio)
necesita los siguientes permisos además de los que ya trae por defecto:

| Política | Para qué |
|---|---|
| `S3DataLakeAccess` | Leer/escribir en el bucket del pipeline (scraping, dataset, modelo) |
| `BedrockInvokeAccess` | Invocar Claude Haiku 4.5 para generar el dataset (sección 5) |
| `PassTrainingRole` | Pasar el rol de entrenamiento a SageMaker al lanzar el job (sección 7) |
| `NotebookConfigSecretReadAccess` | Leer el secreto `slm/notebook-config` de Secrets Manager (bucket, rol, cuenta, región — sección 0.1) |
| `CloudFormationReadAccess` | Respaldo: leer los outputs de los stacks de CDK si el secreto de Secrets Manager aún no existe (sección 0.1) |
| Permisos de CDK (opcional) | Si despliegas la infraestructura (sección 1) desde la propia Terminal de Studio, el rol también necesita los permisos de CloudFormation/IAM/S3 que requiere un `cdk deploy` |

Estos permisos se otorgan **una sola vez**, con una identidad que ya tenga privilegios de
administrador (por ejemplo, AWS CloudShell o la CLI de un administrador; un rol no puede
concederse permisos a sí mismo). Sustituye `<STUDIO_ROLE_NAME>` por el nombre de tu rol de
ejecución de Studio (lo encuentras en SageMaker Studio → User Details → Execution Role):

```bash
# Sustituye <STUDIO_ROLE_NAME> por el nombre de tu rol de ejecución de Studio
# (lo encuentras en SageMaker Studio → User Details → Execution Role)

# 1. Acceso al bucket del datalake
aws iam put-role-policy --role-name <STUDIO_ROLE_NAME> \
  --policy-name S3DataLakeAccess \
  --policy-document '{
    "Version": "2012-10-17",
    "Statement": [{
      "Effect": "Allow",
      "Action": ["s3:GetObject","s3:PutObject","s3:ListBucket","s3:DeleteObject"],
      "Resource": ["arn:aws:s3:::<DATA_BUCKET>","arn:aws:s3:::<DATA_BUCKET>/*"]
    }]
  }'

# 2. Invocar modelos en Bedrock
aws iam put-role-policy --role-name <STUDIO_ROLE_NAME> \
  --policy-name BedrockInvokeAccess \
  --policy-document '{
    "Version": "2012-10-17",
    "Statement": [{
      "Effect": "Allow",
      "Action": ["bedrock:InvokeModel","bedrock:InvokeModelWithResponseStream"],
      "Resource": "arn:aws:bedrock:us-west-2::foundation-model/*"
    }]
  }'

# 3. Pasar el rol de entrenamiento a SageMaker
aws iam put-role-policy --role-name <STUDIO_ROLE_NAME> \
  --policy-name PassTrainingRole \
  --policy-document '{
    "Version": "2012-10-17",
    "Statement": [{
      "Effect": "Allow",
      "Action": "iam:PassRole",
      "Resource": "<TRAINING_ROLE_ARN>",
      "Condition": {"StringEquals": {"iam:PassedToService": "sagemaker.amazonaws.com"}}
    }]
  }'

# 4. Leer el secreto de configuracion del notebook (bucket/rol/cuenta/region, seccion 0.1)
# TrainingStack ya crea la policy administrada 'SlmNotebookConfigSecretReadPolicy' con
# exactamente este permiso, acotado al ARN del secreto: puedes adjuntarla directamente en
# vez de crear una policy inline equivalente.
aws iam attach-role-policy --role-name <STUDIO_ROLE_NAME> \
  --policy-arn <SlmNotebookConfigSecretReadPolicy ARN, output de SlmTrainingStack>

# 5. Respaldo: leer outputs de CloudFormation si el secreto aun no existe (seccion 0.1)
aws iam put-role-policy --role-name <STUDIO_ROLE_NAME> \
  --policy-name CloudFormationReadAccess \
  --policy-document '{
    "Version": "2012-10-17",
    "Statement": [{
      "Effect": "Allow",
      "Action": "cloudformation:DescribeStacks",
      "Resource": "*"
    }]
  }'
```

> **¿Qué es `<DATA_BUCKET>`?** Es el nombre del bucket S3 que crea `SlmDataPipelineStack` al
> hacer `cdk deploy` (sección 1). La sección 0.1 lo lee automáticamente del secreto
> `slm/notebook-config` (o de CloudFormation como respaldo) una vez desplegado.


## Evaluación de modelos candidatos

Este notebook se apoya en la evaluación técnica de modelos y frameworks para inferencia CPU. La tabla siguiente contrasta las decisiones de diseño de este notebook para indicar cada
elección:

| Decisión de este notebook | Evaluación |
|---|---|
| Modelo base: Qwen2.5-1.5B-Instruct (Apache-2.0) | Consistente: Apache-2.0, multilingüe, tamaño adecuado para CPU |
| Técnica de fine-tuning: QLoRA (NF4 4-bit + LoRA r=16/alpha=32) | Rango de `lora-r` (16-64) y cuantización NF4 4-bit |
| Región de entrenamiento: `us-west-2` (GPU) | Uso de instancias `ml.g6.*` (GPU NVIDIA L4), recomendadas para QLoRA de modelos 1B-4B por su mejor costo/hora que la generación anterior (G5/A10G) |
| Catálogo de modelos candidatos para fine-tuning en paralelo | Se selecciona del catálogo de modelos pre-seleccionados: Qwen2.5-1.5B y Qwen3-0.6B |
| Métricas de entrenamiento: duracion, throughput, loss | Disponible en la sección de **memoria pico de GPU/CPU** y **utilización de recursos vía CloudWatch** |
| Evaluación post-entrenamiento de perplexity, tokens/s y latencia | **Aplicada en fases tempranas del ciclo** en donde las métricas aplican directamente sobre los adaptadores recién entrenados, como gate previo a invertir en el pipeline de cuantización |
| Ruta de portabilidad a CPU: GGUF Q4_K_M (ARM) / ONNX INT4 (Intel) | GGUF para Graviton, ONNX/OpenVINO para Intel |
| Instancias de inferencia recomendadas para `mx-central-1` | Por el momento, no se despliega inferencia real en este notebook para la región, solamente se genera el artefacto del modelo portable |

## 0. Cómo ejecutar este notebook

**Prerrequisitos**

- Python 3.10+ y Jupyter (`pip install notebook`).
- Dependencias mínimas para las celdas de **análisis** (las que corren aquí): `matplotlib`,
  `pandas` y `seaborn` (la celda siguiente las verifica e instala si faltan; usadas en las
  Secciones 8.2 y 8.3 para comparar modelos). `json`, `pathlib`, `glob` y `collections` son de
  la librería estándar.
- Para **reproducir** las etapas de AWS (opcional): credenciales de AWS con acceso a S3,
  Bedrock, SageMaker y AgentCore en `us-west-2`, más `boto3`, y las dependencias de cada
  script (`data_pipeline/requirements.txt`, `training/requirements.txt`).
- Para correr el **fine-tuning en paralelo** (Sección 7.4) y la **evaluación post-entrenamiento**
  (Sección 8.3) de forma real: `training/source/requirements.txt` (incluye `psutil` para las
  métricas de memoria) dentro del contenedor de entrenamiento, y
  `training/eval_requirements.txt` (`torch`, `transformers`, `peft`, `accelerate`) en el
  entorno donde se ejecute `evaluate_models.py`.

**Modelo de ejecución: dos tipos de celda**

1. **Celdas de análisis (ejecutables ahora):** leen artefactos que ya están en el repositorio
   (`data_pipeline/processed/…`, `training/metrics_*.json`, `training/eval_metrics_*.json`) y
   no llaman a AWS. Son las de las secciones 0, 3, 4, 5.1, 6.3 y 8. Puedes correrlas de arriba
   a abajo sin configurar nada; si faltan los artefactos de la corrida en paralelo o de la
   evaluación (Secciones 8.2/8.3), lo indican explícitamente en vez de fallar.
2. **Celdas de reproducción (ejecución manual):** contienen los comandos reales del pipeline
   (scraping, Bedrock, SageMaker, AgentCore). **No se ejecutan solas**: imprimen o documentan el
   comando para que lo corras tú de forma controlada. Van marcadas con ⚠️ y el prefijo
   *"EJECUTAR EN TERMINAL"*.

**Scripts externos invocados por las celdas de reproducción.** Para tener visibilidad completa
del proceso, revisa el contenido de estos scripts antes de correrlos (no se ejecutan dentro del
notebook, el notebook solo documenta el comando):

- `data_pipeline/scraping/scrape_cnbv.py`, `data_pipeline/scraping/scrape_banxico.py` (sección 2)
- `data_pipeline/processing/extract_text.py`, `chunk_documents.py`, `build_dataset.py` (secciones 3-5)
- `training/launch_training_job.py`, `training/source/train_qlora.py` (secciones 7 y 7.4)
- `training/evaluate_models.py` (sección 8.3)
- `scripts/05b_run_parallel_finetuning_jobs.sh`, `scripts/06b_evaluate_models.sh` (envuelven a los dos anteriores)

**Orden recomendado:** ejecuta primero la celda de dependencias y la de rutas base (abajo); a
partir de ahí, recorre las secciones en orden. Las celdas de análisis fallan de forma clara si
falta algún artefacto, indicando qué etapa de reproducción hay que correr antes.

> Las celdas de análisis asumen que **este notebook vive en la raíz del repositorio**
> (`aws_summit_slm/`), para que las rutas relativas a `data_pipeline/` y `training/` resuelvan.

In [ ]:
# Dependencias para las celdas de análisis de este notebook.
# matplotlib/pandas/seaborn se usan en la sección 8 (métricas de
# entrenamiento y evaluación de rendimiento, comparando modelos).
import subprocess
import sys as _sys

for _pkg in ("matplotlib", "pandas", "seaborn"):
    try:
        __import__(_pkg)
        print(f"{_pkg} disponible")
    except ImportError:
        print(f"Instalando {_pkg}...")
        subprocess.check_call([_sys.executable, "-m", "pip", "install", "-q", _pkg])
        print(f"{_pkg} instalado.")

### Rutas base del proyecto


In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_PIPELINE = PROJECT_ROOT / "data_pipeline"
PROCESSED = DATA_PIPELINE / "processed"
DATASET_DIR = PROCESSED / "dataset"
TRAINING_DIR = PROJECT_ROOT / "training"

print("Raíz del proyecto:", PROJECT_ROOT)
for p in [DATA_PIPELINE, PROCESSED, DATASET_DIR, TRAINING_DIR]:
    print(f"  {'OK ' if p.exists() else 'NO '} {p.relative_to(PROJECT_ROOT)}")


### 0.1 Configuración desde Secrets Manager (bucket y rol de entrenamiento)

Una vez desplegada la infraestructura (sección 1), `TrainingStack` crea el secreto
`slm/notebook-config` en **AWS Secrets Manager** con el bucket de datos, el ARN del rol de
entrenamiento, la cuenta y la región. Esta celda lee ese secreto con `boto3`, en vez de
depender de un archivo `.env` local o de variables de entorno configuradas a mano: el
acceso se controla por IAM (política `SlmNotebookConfigSecretReadPolicy`, ver "Guía
rápida" más arriba) y el valor nunca queda en disco.

Si el secreto todavía no existe (p.ej. `SlmTrainingStack` no se ha desplegado), la celda
hace un *fallback* automático a leer los mismos valores de los *outputs* de CloudFormation,
y lo indica explícitamente en la salida.

In [ ]:
import json

import boto3

REGION = "us-west-2"
NOTEBOOK_CONFIG_SECRET_NAME = "slm/notebook-config"

session = boto3.Session(region_name=REGION)
secrets_client = session.client("secretsmanager")
cfn = session.client("cloudformation")


def _load_from_secrets_manager():
    """Lee el secreto slm/notebook-config (creado por SlmTrainingStack)."""
    try:
        resp = secrets_client.get_secret_value(SecretId=NOTEBOOK_CONFIG_SECRET_NAME)
    except secrets_client.exceptions.ResourceNotFoundException:
        return None
    except secrets_client.exceptions.ClientError as exc:
        print(f"Aviso: no se pudo leer el secreto '{NOTEBOOK_CONFIG_SECRET_NAME}': {exc}")
        return None
    return json.loads(resp["SecretString"])


def _stack_output(stack_name: str, output_key: str):
    """Respaldo: lee un output de CloudFormation si el secreto no esta disponible."""
    try:
        resp = cfn.describe_stacks(StackName=stack_name)
    except cfn.exceptions.ClientError:
        return None
    outputs = resp["Stacks"][0].get("Outputs", [])
    for o in outputs:
        if o["OutputKey"] == output_key:
            return o["OutputValue"]
    return None


config = _load_from_secrets_manager()
if config:
    print(f"Configuracion leida de Secrets Manager ({NOTEBOOK_CONFIG_SECRET_NAME})")
    data_bucket = config.get("data_bucket_name")
    training_role_arn = config.get("training_role_arn")
    account_id = config.get("account_id")
else:
    print(f"Secreto '{NOTEBOOK_CONFIG_SECRET_NAME}' no encontrado; usando fallback a CloudFormation.")
    data_bucket = _stack_output("SlmDataPipelineStack", "DataBucketName")
    training_role_arn = _stack_output("SlmTrainingStack", "TrainingExecutionRoleArn")
    account_id = session.client("sts").get_caller_identity()["Account"]

print("Account ID:              ", account_id)
print("DATA_BUCKET:             ", data_bucket or "(no detectado; despliega SlmDataPipelineStack en la sección 1)")
print("TRAINING_ROLE_ARN:       ", training_role_arn or "(no detectado; despliega SlmTrainingStack en la sección 1)")

# Si necesitas fijarlos manualmente (p.ej. estas en otra cuenta/region):
# data_bucket = "<nombre-de-tu-bucket>"
# training_role_arn = "<ARN-del-rol-de-entrenamiento>"


## 1. Infraestructura (AWS CDK)

Toda la infraestructura se define en `infra/` con AWS CDK (Python). Son tres stacks fijados a
la región `us-west-2`:

- **`DataPipelineStack`**: un bucket S3 (versionado, cifrado SSE-S3, acceso público bloqueado,
  `enforce_ssl`) con los prefijos lógicos `raw/`, `processed/`, `datasets/`, `models/`; y una
  *managed policy* de lectura/escritura sobre ese bucket, reutilizada por los demás roles.
- **`TrainingStack`**: el rol de ejecución de SageMaker para el training job (acceso al bucket
  del pipeline + CloudWatch Logs/Metrics). El *job* en sí no es un recurso CDK: se lanza como
  ejecución puntual vía `boto3` (sección 7).
- **`AgentRuntimeStack`**: el `agentcore.Runtime` que sirve el agente de prueba, con permisos
  de `s3:GetObject` sobre `models/*` para descargar el adaptador entrenado al arrancar.

El cableado de los tres stacks (nótese la región fija y la dependencia entre stacks):


In [ ]:
# infra/app.py, despliegue con: cd infra && cdk deploy --all
infra_app_snippet = '''
REGION = "us-west-2"  # AgentCore Runtime y cuotas de GPU ml.g6.*

app = cdk.App()
env = cdk.Environment(account=os.getenv("CDK_DEFAULT_ACCOUNT"), region=REGION)

data_pipeline_stack = DataPipelineStack(app, "SlmDataPipelineStack", env=env)
training_stack = TrainingStack(app, "SlmTrainingStack",
                               data_pipeline_stack=data_pipeline_stack, env=env)
agent_runtime_stack = AgentRuntimeStack(app, "SlmAgentRuntimeStack",
                                        data_pipeline_stack=data_pipeline_stack, env=env)
app.synth()
'''
print(infra_app_snippet)

El bucket S3 concentra todas las etapas del pipeline bajo un mismo espacio de nombres,
lo que simplifica los permisos y el rastreo de linaje de datos:

```
s3://<data-bucket>/
├── raw/{cnbv,banxico}/     PDFs + metadata.jsonl        (sección 2)
├── processed/text/         texto extraído               (sección 3)
├── datasets/<job>/         train.jsonl, eval.jsonl      (sección 5-7)
├── code/<job>/             sourcedir.tar.gz             (sección 7)
└── models/<job>/           adaptador LoRA + metrics     (sección 7)
```


La siguiente celda despliega la infraestructura con AWS CDK desde la **Terminal integrada
de SageMaker Studio**. Necesitas tener instalados en el entorno de Studio: AWS CLI
(preinstalado), Node.js (para CDK) y `aws-cdk` (`npm install -g aws-cdk`).

> El rol de ejecución de Studio debe tener permisos para desplegar CloudFormation/IAM/S3
> (ver "Prerrequisitos y permisos IAM" más arriba).

### Despliegue de la infraestructura

Abre la **Terminal integrada de SageMaker Studio** (menú File → New → Terminal) y ejecuta:

```bash
cd ~/aws_summit_slm/infra
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt

# La cuenta se auto-detecta desde las credenciales del rol de Studio;
# solo necesitas fijarla si vas a desplegar en una cuenta distinta:
export CDK_DEFAULT_ACCOUNT=$(aws sts get-caller-identity --query Account --output text)

# Bootstrap CDK (solo la primera vez que usas CDK en esta cuenta/región)
cdk bootstrap aws://$CDK_DEFAULT_ACCOUNT/us-west-2

# Despliega los tres stacks
cdk deploy --all --require-approval never
```

El despliegue toma ~3-5 minutos. Al terminar, CDK imprime los **Outputs** de cada stack.
Los dos valores que necesitarás en las siguientes secciones son:
- `DataBucketName` → el nombre del bucket S3 (para las variables `$DATA_BUCKET`)
- `TrainingExecutionRoleArn` → el ARN del rol de entrenamiento (para `$TRAINING_ROLE_ARN`)

Vuelve a correr la celda de la sección 0.1 después del despliegue para auto-detectar estos
valores dentro del notebook sin tener que copiarlos a mano.

### Cómo obtener los valores de bucket y rol manualmente

Si prefieres obtenerlos por CLI en vez de la celda de la sección 0.1:

```bash
# Nombre del bucket:
aws cloudformation describe-stacks --stack-name SlmDataPipelineStack \
    --query "Stacks[0].Outputs[?OutputKey=='DataBucketName'].OutputValue" \
    --output text --region us-west-2

# ARN del rol de entrenamiento:
aws cloudformation describe-stacks --stack-name SlmTrainingStack \
    --query "Stacks[0].Outputs[?OutputKey=='TrainingExecutionRoleArn'].OutputValue" \
    --output text --region us-west-2
```

> **Importante:** Almacena los valores en variables de entorno en la terminal de Studio para
> no tener que repetirlos en cada comando:
> ```bash
> export DATA_BUCKET="<nombre-de-tu-bucket>"
> export TRAINING_ROLE_ARN="<arn-del-rol>"
> ```


### 1.1 Diseño del datalake para el ciclo continuo de fine-tuning

El bucket no es un simple volcado de PDFs: la normatividad de CNBV y Banxico **cambia con el
tiempo** (nuevas circulares, resoluciones modificatorias, abrogaciones), y el objetivo de negocio
no es entrenar el SLM una sola vez, sino poder **re-entrenarlo periódicamente** conforme el corpus
se actualiza, sin perder trazabilidad de qué versión de los datos produjo qué versión del modelo.
Esto exige tratar el bucket como un **datalake con capas** (zonas), no como una carpeta plana.

**El motor de ese ciclo continuo es `SlmDocumentSyncStack`** (Step Function semanal, domingos
02:00 hora de Ciudad de México): compara el índice público de CNBV/Banxico contra el estado
conocido en el datalake, sube a `raw/` únicamente lo nuevo/modificado, actualiza el catálogo en
DynamoDB, y, si hubo cambios, dispara la preparación de datos. El dataset de fine-tuning y el
entrenamiento (secciones 5-7) son el siguiente eslabón de esa misma cadena, no un proceso aparte.

#### Capas (zonas) del datalake

| Capa | Prefijo S3 | Contenido | Mutabilidad | Quién la escribe |
|---|---|---|---|---|
| **Bronze** (cruda) | `raw/{cnbv,banxico}/` | PDFs originales tal cual se publican | *Upsert* versionado: mismo `s3_key`, nueva versión de S3 en cada cambio real | `sync_documents.py` (semanal) |
| **Silver** (procesada) | `processed/text/`, `processed/chunks.jsonl` | Texto extraído y chunkeado | Se **regenera por completo** en cada corrida a partir de `raw/`; no se versiona por job | `run_data_prep.py`, disparado por el Step Function |
| **Gold** (curada) | `datasets/<job-name>/{train,eval}.jsonl` | Ejemplos SFT usados en un fine-tuning concreto | **Inmutable por convención**: cada corrida escribe un prefijo nuevo, nunca se sobreescribe | `build_dataset.py` / `launch_training_job.py` |
| **Artefactos de código y modelo** | `code/<job-name>/`, `models/<job-name>/`, `models/latest/` | `sourcedir.tar.gz`, adaptador LoRA + `metrics.json` | `<job-name>/` inmutable; `latest/` es un puntero que se **promueve** explícitamente | `launch_training_job.py` / paso de promoción manual |
| **Metadatos operativos** | `sync-runs/latest_summary.json`, tabla DynamoDB `finance_document_catalog` | Resultado de la última sincronización y catálogo consultable de cada documento (`doc_id`, `s3_key`, `s3_version_id`, `status`, `last_synced_at`) | Se sobreescribe/actualiza en cada corrida | `sync_documents.py`, Lambda `update_catalog` |

El **catálogo en DynamoDB** es la pieza que convierte al bucket en un datalake gobernable: en vez
de inferir qué cambió listando objetos de S3 (frágil y costoso a escala), cualquier consumidor
puede preguntar *"¿qué documentos son nuevos o se modificaron desde la última corrida?"* con una
consulta a la tabla. Esto evita el error clásico de particionar por fecha de descarga: los
documentos regulatorios no son un log append-only, son un catálogo de normas vivas que a veces se
modifican o abrogan, así que el estado canónico vive en el catálogo, no en la estructura de
carpetas.

#### Ciclo continuo end-to-end

<!-- TODO: Add diagram -->
<!-- Fuente editable: docs/diagrams/02_ciclo_continuo_datalake.drawio -->

Hoy, el paso de "disparar un nuevo fine-tuning" y la "promoción" del adaptador a `models/latest/`
son decisiones deliberadamente **manuales** (secciones 5-9 de este notebook): entrenar cada vez
que hay un cambio menor en `raw/` no siempre es costo-efectivo. `SlmDocumentSyncStack` deja el
datalake listo (dato fresco + catálogo actualizado) para que ese re-entrenamiento se dispare bajo
un criterio de negocio (p. ej. acumular *N* documentos nuevos, o hacerlo mensual) en lugar de en
cada sincronización semanal.

#### Reglas de gobierno y ciclo de vida

- **Versionado de S3** habilitado en el bucket: cada sobrescritura en `raw/` crea una versión
  nueva en lugar de perder la anterior, lo que permite auditar o revertir un documento fuente.
- **Regla de ciclo de vida** sobre versiones no vigentes de `raw/`: expiran tras un plazo (p. ej.
  180 días) para acotar el costo de almacenamiento sin perder la capacidad de rollback reciente.
- **`datasets/<job-name>/` y `models/<job-name>/` nunca se sobrescriben**: son el registro de
  reproducibilidad de cada fine-tuning (qué datos exactos produjeron qué adaptador); solo
  `models/latest/` se actualiza, y únicamente como parte de una promoción explícita.
- **Convención de nombres compartida**: el mismo `job-name`
  (`cnbv-banxico-qwen25-1-5b-qlora-<timestamp>`) se usa en `code/`, `datasets/` y `models/`, de
  forma que los tres artefactos de una corrida de entrenamiento son trivialmente correlacionables
  por prefijo, sin necesidad de una tabla adicional.
- **El catálogo DynamoDB es la fuente de verdad del estado actual** del corpus (qué documentos
  existen, en qué `s3_version_id`); la capa *silver*/*gold* se puede regenerar por completo en
  cualquier momento a partir de `raw/` + catálogo, lo que hace innecesario un merge incremental
  frágil en `processed/`.


In [ ]:
# Referencia de las capas del datalake. No llama a AWS.
DATALAKE_LAYERS = {
    "bronze_raw": {
        "prefix": "raw/{source}/",
        "contenido": "PDFs originales de CNBV/Banxico",
        "mutabilidad": "upsert versionado (mismo key, nueva versión de S3)",
        "escritor": "data_pipeline/sync/sync_documents.py (semanal, via Step Function)",
    },
    "silver_processed": {
        "prefix": "processed/text/, processed/chunks.jsonl",
        "contenido": "Texto extraido (pypdf) y chunkeado por parrafos",
        "mutabilidad": "se regenera por completo desde raw/ en cada corrida",
        "escritor": "data_pipeline/processing/run_data_prep.py",
    },
    "gold_dataset": {
        "prefix": "datasets/<job-name>/{train,eval}.jsonl",
        "contenido": "Pares instruccion/respuesta (SFT) usados en un fine-tuning concreto",
        "mutabilidad": "inmutable: un prefijo nuevo por corrida, nunca se sobreescribe",
        "escritor": "data_pipeline/processing/build_dataset.py + training/launch_training_job.py",
    },
    "model_artifacts": {
        "prefix": "code/<job-name>/, models/<job-name>/, models/latest/",
        "contenido": "sourcedir.tar.gz, adaptador LoRA + metrics.json",
        "mutabilidad": "<job-name>/ inmutable; latest/ se promueve explícitamente",
        "escritor": "training/launch_training_job.py + paso de promocion manual",
    },
    "catalog_metadata": {
        "prefix": "sync-runs/latest_summary.json + tabla DynamoDB finance_document_catalog",
        "contenido": "Estado de la ultima sincronizacion y catalogo por documento (doc_id, s3_version_id, status)",
        "mutabilidad": "se actualiza (upsert) en cada corrida",
        "escritor": "data_pipeline/sync/sync_documents.py + catalog_lambda/update_catalog.py",
    },
}

for layer, meta in DATALAKE_LAYERS.items():
    print(f"[{layer}]")
    for k, v in meta.items():
        print(f"  {k:>12}: {v}")
    print()


## 2. Adquisición de datos: scraping de CNBV y Banxico

Los datos de valor son los **documentos normativos** publicados por ambas autoridades. Se
obtienen con dos scrapers dedicados, porque la estructura de cada sitio es distinta.

### 2.1 CNBV: `data_pipeline/scraping/scrape_cnbv.py`

La CNBV publica su normatividad por **sector supervisado** (9 sectores: banca múltiple,
bursátil, sociedades de inversión, uniones de crédito, banca de desarrollo, sector popular,
etc.). Cada página de "Normatividad" renderiza una **tabla HTML** con una fila por norma
(nombre, tipo, fecha de publicación en el DOF, sectores aplicables) y un enlace directo al PDF.
Además, por cada norma con `doc_id` válido se consulta un endpoint AJAX de *Resoluciones y
Anexos* para descubrir PDFs relacionados (resoluciones modificatorias y anexos), que también
enriquecen el corpus.

### 2.2 Banxico: `data_pipeline/scraping/scrape_banxico.py`

Banxico expone un **índice cronológico** (~300 circulares/disposiciones desde 1969). El
scraper trabaja en dos pasos: primero extrae los enlaces de cada página del índice (cada
circular), y luego, en cada una, localiza el enlace directo al PDF real.

### 2.3 Detalle técnico: certificados TLS (`ca_bundle.py`)

Ambos sitios presentan **cadenas de certificados incompletas** (no envían el certificado
intermedio de su CA). Los navegadores lo toleran vía AIA, pero el módulo `ssl` de Python no,
provocando `CERTIFICATE_VERIFY_FAILED`. La solución fue construir en tiempo de ejecución un
*bundle* CA que combina `certifi` con los intermedios faltantes (guardados en
`data_pipeline/certs/`), y pasarlo como `verify=` a `requests`.


⚠️ **ACCIÓN REQUERIDA — Ejecutar en terminal de SageMaker Studio (no en esta celda)**

La siguiente celda **NO se auto-ejecuta**. Los comandos de scraping deben correrse manualmente en la **Terminal integrada de SageMaker Studio** (menú File → New → Terminal).

**Tiempo estimado:** 15-25 minutos

**Antes de ejecutar:** asegúrate de tener definida la variable `$DATA_BUCKET` con el nombre de tu bucket (obtenido en la sección 1):

```bash
export DATA_BUCKET="<data-bucket>"
```

Podemos realizar la verificación del valor del DATA_BUCKET con el siguiente comando:

```bash
echo $DATA_BUCKET
```

Los siguientes comandos permiten automatizar el proceso de descarga de archivos regulatorios desde una lista de sitios web de información pública oficial actualizada a partir del cual nos apoyaremos para establecer una base de datos para llevar a cabo el proceso de fine-tuning.

In [ ]:
# ⚠️ EJECUTAR EN TERMINAL DE SAGEMAKER STUDIO (no en esta celda)
# Tiempo estimado: 15-25 minutos
#
# Primero, define tu bucket (obtén el nombre con el comando de la sección 1):
#   export DATA_BUCKET="<nombre-de-tu-bucket>"
#
# Luego ejecuta:
#   cd ~/aws_summit_slm/data_pipeline
#   pip install -r requirements.txt
#   python scraping/scrape_cnbv.py --out-dir ./raw_cnbv
#   python scraping/scrape_banxico.py --out-dir ./raw_banxico
#   aws s3 sync ./raw_cnbv    s3://$DATA_BUCKET/raw/cnbv/
#   aws s3 sync ./raw_banxico s3://$DATA_BUCKET/raw/banxico/
print("⚠️  Ejecutar en terminal de Studio. Ver instrucciones arriba.")


**Contrato de salida (`metadata.jsonl`)**: una línea JSON por documento descargado, con
la fuente, el sector, el tipo de norma, la fecha de publicación en el DOF, la URL original y
la ruta local. Este manifiesto es la base de trazabilidad del corpus.


## 3. Extracción de texto: `processing/extract_text.py`

Cada PDF se convierte a texto plano con **`pypdf`**, aplicando una limpieza básica
(normalización de espacios, colapso de saltos de página y de líneas repetidas). Los documentos
con menos de **200 caracteres** se descartan, ya que casi siempre son PDFs escaneados sin OCR, que no
aportan texto entrenable. El resultado se escribe en `processed/text/<fuente>/<slug>.txt` junto
con un `manifest_<fuente>.jsonl` que registra número de páginas y de caracteres por documento.


⚠️ **Antes de ejecutar la celda de análisis de abajo**, debes haber corrido los
siguientes comandos en la **terminal de SageMaker Studio**:

In [ ]:
# ⚠️ EJECUTAR EN TERMINAL DE SAGEMAKER STUDIO (no en esta celda)
# Tiempo estimado: 2-5 min | Llama a AWS: No
#
#   cd ~/aws_summit_slm/data_pipeline
#   python processing/extract_text.py --raw-dir ./raw_cnbv    --source cnbv    --out-dir ./processed/text
#   python processing/extract_text.py --raw-dir ./raw_banxico --source banxico --out-dir ./processed/text
print("⚠️  Ejecutar en terminal de Studio. Ver instrucciones arriba.")


**Tiempo estimado:** 2-5 minutos | **Llama a AWS:** No (procesa archivos locales)

Sustituye `$DATA_BUCKET` con el nombre de tu bucket (obtenido en la sección 1) si deseas subir el resultado a S3 después:

In [ ]:
!aws s3 sync ./processed/text s3://$DATA_BUCKET/processed/text/

In [ ]:
cnbv_dir = PROCESSED / "text" / "cnbv"
banxico_dir = PROCESSED / "text" / "banxico"

n_cnbv = len(list(cnbv_dir.glob("*.txt"))) if cnbv_dir.exists() else 0
n_banxico = len(list(banxico_dir.glob("*.txt"))) if banxico_dir.exists() else 0

print(f"Documentos de texto CNBV:    {n_cnbv}")
print(f"Documentos de texto Banxico: {n_banxico}")
print(f"Total corpus:                {n_cnbv + n_banxico}")


## 4. Chunking: `processing/chunk_documents.py`

Los documentos regulatorios son largos (algunos con cientos de páginas, como la Circular Única
de Bancos). Para poder usarlos como **contexto acotado** al generar ejemplos de instrucción, se
fragmentan en *chunks* de tamaño manejable.

Estrategia: dividir por párrafos (doble salto de línea) y acumular hasta un objetivo de
**~3 000 caracteres** (tope duro de 4 500), sin cortar párrafos a la mitad; los fragmentos de
menos de 400 caracteres se descartan por ser ruido (encabezados sueltos, etc.).


⚠️ **Antes de ejecutar la celda de análisis de abajo**, debes haber corrido el
siguiente comando en la **terminal de SageMaker Studio**:

In [ ]:
# ⚠️ EJECUTAR EN TERMINAL DE SAGEMAKER STUDIO (no en esta celda)
# Tiempo estimado: 1-2 min | Llama a AWS: No
#
#   cd ~/aws_summit_slm/data_pipeline
#   python processing/chunk_documents.py --text-dir ./processed/text --out ./processed/chunks.jsonl
print("⚠️  Ejecutar en terminal de Studio. Ver instrucciones arriba.")


**Tiempo estimado:** 1-2 minutos | **Llama a AWS:** No (procesa archivos locales)

In [ ]:
import json

chunks_path = PROCESSED / "chunks.jsonl"
n_chunks = 0
by_source = {"cnbv": 0, "banxico": 0}
sample_chunk = None
if chunks_path.exists():
    with open(chunks_path, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            n_chunks += 1
            by_source[rec["source"]] = by_source.get(rec["source"], 0) + 1
            if sample_chunk is None:
                sample_chunk = rec

print(f"Chunks totales: {n_chunks}")
print(f"Por fuente: {by_source}")

if sample_chunk:
    print(f"\nEjemplo de chunk ({sample_chunk['source']}, {sample_chunk['original_pdf']}):")
    print(sample_chunk["text"][:600], "...")


## 5. Construcción del dataset de instrucción (SFT): `processing/build_dataset.py`

Este es el corazón de la preparación de datos. El corpus por sí solo es texto normativo crudo;
para el fine-tuning necesitamos **pares instrucción : respuesta** que reflejen las tareas reales
del negocio. Se generan de forma sintética pero *grounded*: para cada chunk seleccionado se
invoca **Amazon Bedrock (Claude Haiku 4.5)** con la instrucción de producir
ejemplos basados **únicamente** en el contenido del fragmento.

Los ejemplos cubren las tres tareas del caso de uso:

1. **Resumen estructurado** de la disposición (objeto, sujetos obligados, plazos, sanciones).
2. **Extracción de obligaciones** de cumplimiento en formato de checklist accionable.
3. **Clasificación regulatoria** (sector / tipo de norma) con justificación.
4. *(complementaria)* **Explicación en lenguaje sencillo** de un requisito.

Decisiones de diseño relevantes:

- **Balanceo por documento** (`--max-chunks-per-doc`): evita que documentos enormes (p.ej. la
  Circular Única de Bancos, con cientos de chunks) dominen el dataset.
- **Paralelismo con `ThreadPoolExecutor`** y un cliente `boto3` por hilo, con *backoff*
  exponencial ante `ThrottlingException`.
- **Formato de chat** (`system`/`user`/`assistant`) serializado en JSONL, compatible con el
  *chat template* de Qwen2.5 y con `trl.SFTTrainer`.
- **Split 90/10** en `train.jsonl` / `eval.jsonl`.

El *system prompt* que se fija en cada ejemplo (y que después reutiliza el agente en producción):


In [ ]:
SYSTEM_PROMPT_SLM = (
    "Eres un asistente especializado en cumplimiento regulatorio financiero "
    "mexicano (CNBV y Banxico). Ayudas a evaluar carpetas de cumplimiento y a "
    "estructurar documentos regulatorios de forma precisa y concisa, citando "
    "el fundamento normativo cuando sea posible."
)
print(SYSTEM_PROMPT_SLM)


⚠️ **ACCIÓN REQUERIDA — Ejecutar en terminal de SageMaker Studio (no en esta celda)**
La generación del dataset **consume Amazon Bedrock** (Claude Haiku 4.5) y tiene un
costo asociado (~$3-5 USD dependiendo del número de chunks).

**Tiempo estimado:** 30-60 minutos

**Costo:** ~$3-5 USD (llamadas a Bedrock)
Ejecuta los siguientes comandos en la **Terminal de SageMaker Studio**:

In [ ]:
# ⚠️ EJECUTAR EN TERMINAL DE SAGEMAKER STUDIO (no en esta celda)
# Tiempo estimado: 30-60 minutos | Costo: ~$3-5 USD (Bedrock)
#
# Asegúrate de tener definida tu variable de entorno:
#   export DATA_BUCKET="<nombre-de-tu-bucket>"
#
# Luego ejecuta:
#   cd ~/aws_summit_slm/data_pipeline
#   python processing/build_dataset.py \
#       --chunks ./processed/chunks.jsonl \
#       --out-dir ./processed/dataset \
#       --max-chunks-per-doc 10 \
#       --num-examples-per-chunk 2 \
#       --region us-west-2
#
# Para subir a S3:
#   aws s3 sync ./processed/dataset s3://$DATA_BUCKET/datasets/
print("⚠️  Ejecutar en terminal de Studio. Ver instrucciones arriba.")


### 5.1 Inspección del dataset generado

Los artefactos ya generados están en `data_pipeline/processed/dataset/`. Cargamos los conteos y
un ejemplo real para verificar el formato de chat.


In [ ]:
def count_lines(path):
    if not path.exists():
        return 0
    with open(path, encoding="utf-8") as f:
        return sum(1 for _ in f)

train_path = DATASET_DIR / "train.jsonl"
eval_path = DATASET_DIR / "eval.jsonl"
meta_path = DATASET_DIR / "dataset_with_metadata.jsonl"

n_train = count_lines(train_path)
n_eval = count_lines(eval_path)
n_total = count_lines(meta_path)

print(f"Ejemplos de entrenamiento (train.jsonl): {n_train}")
print(f"Ejemplos de evaluación   (eval.jsonl):   {n_eval}")
print(f"Total con metadata:                      {n_total}")
if n_total:
    print(f"Split efectivo:  train {n_train/(n_train+n_eval):.1%} / eval {n_eval/(n_train+n_eval):.1%}")


In [ ]:
# Distribución por fuente (CNBV vs Banxico) en el dataset con metadata
from collections import Counter
import json

src_counter = Counter()
if meta_path.exists():
    with open(meta_path, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            src_counter[rec.get("_source", "desconocido")] += 1
print("Ejemplos por fuente:", dict(src_counter))


In [ ]:
# Un ejemplo real del dataset (formato de chat system/user/assistant)
if train_path.exists():
    with open(train_path, encoding="utf-8") as f:
        example = json.loads(f.readline())
    for msg in example["messages"]:
        print(f"### {msg['role'].upper()}")
        print(msg["content"][:700])
        print()


## 6. Selección del modelo base y de la técnica de fine-tuning

Esta decisión está condicionada por dos restricciones del enunciado: **baja latencia / bajo
costo** y **portabilidad a `mx-central-1` (solo CPU Intel/ARM, sin GPU)**.

### 6.1 Modelo base: **Qwen2.5-1.5B-Instruct**

| Criterio | Por qué Qwen2.5-1.5B-Instruct |
|---|---|
| **Licencia** | Apache-2.0 (uso comercial libre). Qwen2.5-3B/7B usan licencia *research* (no comercial); Llama-3.2-1B/3B son *gated* con cláusulas de atribución. |
| **Idiomas** | 29 idiomas, incluido español. |
| **Tamaño** | 1.54 B parámetros (~3 GB en fp16, ~1 GB en int4): compatible con baja latencia en CPU. |
| **Contexto** | 32K tokens, de sobra para chunks de circulares. |
| **Cuantización CPU** | Ecosistema maduro: **GGUF** (llama.cpp con optimizaciones ARM/KleidiAI en Graviton) e **IR de OpenVINO** INT4/INT8 para CPU Intel. Clave para `mx-central-1`. |
| **Fine-tuning** | Soporte de primera clase en `transformers` + `peft` + `trl` + `bitsandbytes`. |

### 6.2 Técnica: **QLoRA**

- Cuantización **NF4 de 4 bits** del modelo base + **adaptadores LoRA** entrenables (rank 16,
  alpha 32) sobre las proyecciones de atención y MLP.
- Reduce la memoria de entrenamiento ~70%: permite entrenar en **una sola GPU** `ml.g6.2xlarge`
  (NVIDIA L4, 24 GB; mismo VRAM que la generación G5/A10G pero ~15-20% más económica por hora).
- El artefacto resultante (el adaptador) pesa **decenas de MB**, y se puede **fusionar** con el
  modelo base y re-cuantizar a GGUF/OpenVINO para el despliegue en CPU (sección 10).

Las capas objetivo del adaptador LoRA:


In [ ]:
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",   # atención
    "gate_proj", "up_proj", "down_proj",       # MLP
]
print("Módulos objetivo de LoRA:", LORA_TARGET_MODULES)


### 6.3 Catálogo de modelos para fine-tuning en paralelo (validación contra `docs/technical_documentation.md`)

El requisito de negocio pide poder comparar el costo/beneficio de entrenar varios modelos
candidatos, no solo Qwen2.5-1.5B-Instruct. `training/model_catalog.py` define un catálogo con
los modelos de la Sección 3.1 de `docs/technical_documentation.md` ("Models Validated for CPU
Inference") que **además** tienen una ruta de fine-tuning QLoRA/LoRA documentada en la
Sección 7.1 ("Supported Approaches"):

| Clave | Modelo | Parametros | Licencia | Modulos LoRA | Instancia recomendada |
|---|---|---|---|---|---|
| `qwen2.5-1.5b` | Qwen/Qwen2.5-1.5B-Instruct | 1.5B | Apache-2.0 | q/k/v/o_proj, gate/up/down_proj | `ml.g6.xlarge` |
| `qwen3-0.6b` | Qwen/Qwen3-0.6B | 0.6B | Apache-2.0 | q/k/v/o_proj, gate/up/down_proj | `ml.g6.xlarge` |

Ambos modelos comparten la misma arquitectura de proyecciones de atención y MLP separadas,
por lo que usan el mismo conjunto de `lora_target_modules` en `training/model_catalog.py`.

In [ ]:
import sys as _sys
_sys.path.insert(0, str(TRAINING_DIR))
from model_catalog import list_model_specs

for spec in list_model_specs():
    print(f"[{spec.key}] {spec.model_id}  ({spec.params_b}B, {spec.license})")
    print(f"    lora_target_modules:      {spec.lora_target_modules}")
    print(f"    instancia recomendada:    {spec.recommended_instance_type}")
    print(f"    ruta de inferencia CPU:   {spec.cpu_inference_path}")
    print()

## 7. Fine-tuning con QLoRA en SageMaker

El entrenamiento se ejecuta como un **SageMaker Training Job** usando el **HuggingFace PyTorch
Training DLC** (PyTorch 2.9 + Transformers 5.3, imagen GPU), sobre el cual se instalan
`peft`, `trl`, `bitsandbytes`, `accelerate` y `datasets` vía `requirements.txt`.

Dos piezas:

- **`training/launch_training_job.py`**: orquestador local (`boto3`, sin el SDK de alto nivel
  de SageMaker). Empaqueta `training/source/` en `sourcedir.tar.gz`, lo sube a S3, sube los
  datasets, llama a `create_training_job` y espera a que termine, recolectando métricas.
- **`training/source/train_qlora.py`**: *entry point* que corre dentro del contenedor: carga
  el modelo en 4 bits, aplica LoRA, entrena con `trl.SFTTrainer` y guarda el adaptador + un
  `metrics.json`.

### 7.1 Hiperparámetros del job


In [ ]:
hyperparameters = {
    "model-id": "Qwen/Qwen2.5-1.5B-Instruct",
    "epochs": 3,
    "learning-rate": 2e-4,
    "per-device-train-batch-size": 4,
    "per-device-eval-batch-size": 4,
    "gradient-accumulation-steps": 4,   # batch efectivo = 4 * 4 = 16
    "max-seq-length": 1024,
    "lora-r": 16,
    "lora-alpha": 32,
    "lora-dropout": 0.05,
    "seed": 42,
}
INSTANCE_TYPE = "ml.g6.2xlarge"  # 1x NVIDIA L4 24GB
for k, v in hyperparameters.items():
    print(f"  {k:>32} = {v}")
print(f"\n  instancia = {INSTANCE_TYPE}")


### 7.2 Configuración de cuantización 4-bit (dentro de `train_qlora.py`)

La cuantización NF4 con doble cuantización y cómputo en `bfloat16` es la esencia de QLoRA:


### Comparativo: fine-tuning completo, LoRA y QLoRA

**LoRA (Low-Rank Adaptation)**, propuesto por Hu et al. en
["LoRA: Low-Rank Adaptation of Large Language Models"](https://arxiv.org/abs/2106.09685)
(Microsoft, 2021), parte de una observación empírica: cuando un modelo pre-entrenado se adapta a
una tarea nueva, la actualización de sus pesos suele tener **rango intrínseco bajo**. En vez de
actualizar la matriz de pesos completa $W_0 \in \mathbb{R}^{d\times k}$ de cada capa, LoRA la
congela y aprende una actualización $\Delta W = BA$, donde $B \in \mathbb{R}^{d\times r}$ y
$A \in \mathbb{R}^{r\times k}$, con un rango $r \ll \min(d, k)$ (en este proyecto, `lora-r = 16`
frente a dimensiones ocultas de miles). Solo $A$ y $B$ son entrenables. Los autores reportan que,
sobre GPT-3 175B, esto reduce los parámetros entrenables **10,000 veces** y el requerimiento de
memoria de GPU **3 veces**, con calidad igual o mejor que el fine-tuning completo y **sin
latencia adicional en inferencia** (los adaptadores se pueden fusionar con $W_0$).

**QLoRA**, de Dettmers et al. en
["QLoRA: Efficient Finetuning of Quantized LLMs"](https://arxiv.org/abs/2305.14314)
(Universidad de Washington, 2023), lleva la idea un paso más allá: además de congelar $W_0$,
**la cuantiza a 4 bits** y solo mantiene los adaptadores LoRA en mayor precisión (`bfloat16`).
El paper reporta permitir el fine-tuning de un modelo de **65B parámetros en una sola GPU de
48GB**, preservando el desempeño del fine-tuning en 16 bits completo. Introduce tres innovaciones
que se usan directamente en `train_qlora.py` (sección 7.2):

1. **NF4 (4-bit NormalFloat):** un tipo de dato de 4 bits diseñado para representar pesos que
   siguen una distribución normal (como los de una red ya entrenada) de forma óptima en términos
   de teoría de la información, más preciso que un INT4 genérico para este caso de uso.
2. **Doble cuantización:** cuantiza también las *constantes* de cuantización, reduciendo aún más
   la huella de memoria promedio.
3. **Optimizadores paginados:** usan la memoria unificada de NVIDIA para absorber picos de
   memoria durante el entrenamiento (p. ej. al calcular gradientes de secuencias largas) sin
   provocar *out-of-memory*.

| Aspecto | Fine-tuning completo | LoRA | QLoRA |
|---|---|---|---|
| Pesos base | Se actualizan todos | Congelados en su precisión original | Congelados y cuantizados a 4-bit (NF4) |
| Parámetros entrenables | 100% del modelo | Adaptadores de bajo rango (típicamente <1%) | Igual que LoRA |
| Memoria de GPU | Muy alta (pesos + gradientes + estados del optimizador para todo el modelo) | Alta (pesos base en fp16/bf16 + adaptadores) | Baja (pesos base en 4-bit + adaptadores) |
| Riesgo de *catastrophic forgetting* | Mayor | Menor (el conocimiento base queda intacto en $W_0$) | Menor, igual que LoRA |
| Latencia en inferencia | Sin cambios | Sin cambios si se fusiona el adaptador | Sin cambios si se fusiona el adaptador (ver sección 10) |
| Adecuado para... | Cambios de dominio muy profundos, con mucho cómputo disponible | GPUs con memoria moderada | GPUs con memoria limitada (este proyecto: 1x NVIDIA L4 24GB en `ml.g6.2xlarge`) |

**Por qué QLoRA para este proyecto:** con un modelo de 1.5B parámetros y una sola GPU
`ml.g6.2xlarge` (NVIDIA L4, 24GB), un fine-tuning completo en `bfloat16` ya sería viable en memoria,
pero QLoRA reduce aún más el consumo, permite `per-device-train-batch-size` más alto y deja margen
para escalar a modelos base más grandes sin cambiar de tipo de instancia, relevante dado el
requisito de mantener el costo bajo.

In [ ]:
# Extracto de training/source/train_qlora.py
bnb_config_repr = """
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=LORA_TARGET_MODULES,
    bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)   # solo el adaptador es entrenable
"""
print(bnb_config_repr)


### El rol de ejecución de SageMaker (`TrainingExecutionRoleArn`)

Todo Training Job de SageMaker necesita un **rol de ejecución de IAM** (`RoleArn` en el parámetro
[`CreateTrainingJob`](https://docs.aws.amazon.com/sagemaker/latest/dg/API_CreateTrainingJob.html)):
SageMaker *asume* ese rol para, en nombre del job, leer el dataset de entrada, descargar la imagen
del contenedor, escribir logs/métricas a CloudWatch y subir el artefacto del modelo resultante a
S3. Sin este rol el `create_training_job` de `launch_training_job.py` (sección 7.3) falla por
falta de permisos.

En este proyecto el rol **no se crea a mano**: lo provisiona `TrainingStack` (`infra/stacks/training_stack.py`)
como parte del `cdk deploy --all` de la sección 1, con:

- **Trust policy** que permite únicamente al *service principal* `sagemaker.amazonaws.com` asumirlo.
- **`AmazonSageMakerFullAccess`** (managed policy de AWS) para las operaciones estándar del job.
- **`pipeline_access_policy`**: la misma policy que usan las tareas Fargate de sincronización
  documental (sección 1.1), para leer/escribir en el datalake `datalake-<account-id>` sin
  duplicar permisos entre stacks.
- Permisos explícitos de CloudWatch Logs/Metrics (ya cubiertos por la managed policy, pero
  declarados aparte para poder acotar el alcance en el futuro sin depender de ella).

El ARN resultante se expone como el output `TrainingExecutionRoleArn` del stack, y es
exactamente el valor que se pasa a `--role-arn` en el comando de la sección 7.3:

```bash
aws cloudformation describe-stacks --stack-name SlmTrainingStack \
    --query "Stacks[0].Outputs[?OutputKey=='TrainingExecutionRoleArn'].OutputValue" \
    --output text
```

### 7.3 Lanzamiento del training job

> ⚠️ **ACCIÓN REQUERIDA — Ejecutar en terminal de SageMaker Studio (no en esta celda)**
>
> Este paso lanza un **SageMaker Training Job** que usa una instancia GPU `ml.g6.2xlarge`.
> El job tarda en completar y tiene un costo asociado.
>
> **Tiempo estimado:** 60-100 minutos | **Costo:** ~$2 USD
>
> Ejecuta los siguientes comandos en la **Terminal de SageMaker Studio**:
>
> ```bash
> # Sustituye los valores con los outputs de CloudFormation (sección 1):
> export DATA_BUCKET="<nombre-de-tu-bucket>"
> export TRAINING_ROLE_ARN="<ARN-del-rol-de-entrenamiento>"
>
> cd ~/aws_summit_slm/training
> pip install -r requirements.txt
> python launch_training_job.py \
>     --bucket $DATA_BUCKET \
>     --role-arn $TRAINING_ROLE_ARN \
>     --region us-west-2 \
>     --instance-type ml.g6.2xlarge \
>     --epochs 3
> ```
>
> **¿Cómo obtener los valores?** (si no los tienes a la mano):
> ```bash
> # Nombre del bucket:
> aws cloudformation describe-stacks --stack-name SlmDataPipelineStack \
>     --query "Stacks[0].Outputs[?OutputKey=='DataBucketName'].OutputValue" \
>     --output text --region us-west-2
>
> # ARN del rol de entrenamiento:
> aws cloudformation describe-stacks --stack-name SlmTrainingStack \
>     --query "Stacks[0].Outputs[?OutputKey=='TrainingExecutionRoleArn'].OutputValue" \
>     --output text --region us-west-2
> ```

El lanzador espera a que el job termine (`describe_training_job` en *polling*) y guarda un
archivo `metrics_<job-name>.json` combinando las métricas del contenedor (loss, throughput)
con las de SageMaker (duración facturable, tipo de instancia). El artefacto del modelo
(adaptador LoRA) queda en `s3://<bucket>/models/<job-name>/`.


In [ ]:
# ⚠️ EJECUTAR EN TERMINAL DE SAGEMAKER STUDIO (no en esta celda)
# Tiempo estimado: 60-100 minutos | Costo: ~$2 USD
#
# Sustituye los valores:
#   export DATA_BUCKET="<nombre-de-tu-bucket>"
#   export TRAINING_ROLE_ARN="<ARN-del-rol-de-entrenamiento>"
#
#   cd ~/aws_summit_slm/training
#   pip install -r requirements.txt
#   python launch_training_job.py \
#       --bucket $DATA_BUCKET \
#       --role-arn $TRAINING_ROLE_ARN \
#       --region us-west-2 \
#       --instance-type ml.g6.2xlarge \
#       --epochs 3
print("⚠️  Ejecutar en terminal de Studio. Ver instrucciones arriba.")


### 7.4 Fine-tuning en paralelo de múltiples modelos candidatos

Además del flujo single-job de la sección 7.3 (que sigue funcionando igual), 
`training/launch_training_job.py` soporta lanzar **un SageMaker Training Job independiente por
cada modelo del catálogo de la sección 6.3**, todos sobre el mismo dataset (`train.jsonl` /
`eval.jsonl`), y esperarlos **concurrentemente** con un `ThreadPoolExecutor`. Esto permite
comparar, bajo las mismas condiciones de datos e hiperparametros, cuanto tiempo/memoria/costo
toma entrenar cada modelo y que tan bien queda el `eval_loss` de cada uno (sección 8.2).

Cada job:

1. Usa la instancia recomendada de su ficha en el catálogo (`ml.g6.xlarge` para ambos modelos,
   ≤1.5B), salvo que se fije `--instance-type` para forzar la misma instancia en todos.
2. Recibe sus propios `--model-id` y `--lora-target-modules` (Sección 6.3: ambos modelos usan
   las proyecciones de atención y MLP separadas).
3. Al terminar, descarga su `metrics.json` (con las métricas de memoria de la Sección 8.2:
   `gpu_peak_allocated_mb`, `gpu_peak_reserved_mb`, `cpu_rss_after_train_mb`,
   `adapter_size_mb`) y lo combina con `describe_training_job` (duracion, costo estimado) y
   con la utilización de CPU/GPU de CloudWatch (`resource_utilization`).

El resultado se guarda en `training/metrics_<job_name>.json` por modelo, mas un archivo
consolidado `training/parallel_run_<timestamp>.json` con el resumen de la corrida completa.

```bash
cd training
python launch_training_job.py \
    --bucket $DATA_BUCKET \
    --role-arn $TRAINING_ROLE_ARN \
    --region us-west-2 \
    --model-keys qwen2.5-1.5b,qwen3-0.6b \
    --max-workers 2 \
    --epochs 3
```

Equivalente via el script de reproduccion (`scripts/05b_run_parallel_finetuning_jobs.sh`):

```bash
cd scripts
MODEL_KEYS=qwen2.5-1.5b,qwen3-0.6b ./05b_run_parallel_finetuning_jobs.sh
```

> **Nota de cuota/costo:** esto lanza 2 instancias GPU **simultáneamente**. Verifica que la
> cuota de servicio de SageMaker (`ml.g6.xlarge for training job usage`) sea suficiente para
> el numero de jobs en paralelo, o reduce `--max-workers`/`--model-keys`.

## 8. Análisis de métricas del entrenamiento

Esta sección cubre tres análisis, de menor a mayor alcance:

- **8.1 Corrida individual (legacy)**: el análisis original de un único job (Qwen2.5-1.5B),
  leyendo `training/metrics_*.json`.
- **8.2 Comparativo multi-modelo en paralelo**: si existe una corrida de
  `training/launch_training_job.py --model-keys` (Sección 7.4), compara tiempo de
  entrenamiento, memoria (GPU/CPU) y loss entre los modelos entrenados, con `pandas` y
  `seaborn`.
- **8.3 Evaluación de rendimiento post-entrenamiento**: carga los resultados de
  `training/evaluate_models.py` (perplexity, tokens/segundo, latencia) y los visualiza con
  `seaborn` para apoyar la decisión de qué modelo promover.

### 8.1 Corrida individual (legacy)

Cargamos el archivo de métricas real producido por el job single-job ejecutado originalmente
(`training/metrics_*.json`, Sección 7.3). Este análisis sigue funcionando igual que antes;
es el punto de partida de comparación para el análisis multi-modelo de la Sección 8.2.

In [ ]:
import glob
import json

metric_files = sorted(glob.glob(str(TRAINING_DIR / "metrics_*.json")))
print("Archivos de métricas encontrados:")
for m in metric_files:
    print("  -", Path(m).name)

metrics = {}
if metric_files:
    with open(metric_files[-1], encoding="utf-8") as f:
        metrics = json.load(f)
metrics


In [ ]:
# Resumen legible de las métricas clave.
# wall-clock incluye aprovisionamiento y descarga del modelo base;
# training_duration_seconds es solo el bucle de entrenamiento (SFTTrainer).
if metrics:
    dur = metrics.get("duration_seconds") or 0
    train_loop = metrics.get("training_duration_seconds") or 0
    billable = metrics.get("billable_time_seconds") or 0
    print("=" * 60)
    print(f" Job:                {metrics.get('training_job_name')}")
    print(f" Estado:             {metrics.get('training_job_status')}")
    print(f" Instancia:          {metrics.get('instance_type')}")
    print("-" * 60)
    print(f" Modelo base:        {metrics.get('base_model_id', 'N/A')}")
    print(f" Ejemplos train:     {metrics.get('num_train_examples', 'N/A')}")
    print(f" Ejemplos eval:      {metrics.get('num_eval_examples', 'N/A')}")
    print(f" Épocas:             {metrics.get('epochs', 'N/A')}")
    print(f" LoRA r / alpha:     {metrics.get('lora_r', 'N/A')} / {metrics.get('lora_alpha', 'N/A')}")
    print("-" * 60)
    print(f" Wall-clock del job: {dur:.0f} s  (~{dur/60:.1f} min)")
    print(f" Entrenamiento puro: {train_loop:.0f} s  (~{train_loop/60:.1f} min)")
    print(f" Tiempo facturable:  {billable} s")
    print(f" Throughput:         {metrics.get('train_samples_per_second', 'N/A')} samples/s")
    train_loss = metrics.get('final_train_loss')
    eval_loss = metrics.get('eval_loss')
    print(f" Train loss final:   {f'{train_loss:.4f}' if train_loss is not None else 'N/A'}")
    print(f" Eval loss:          {f'{eval_loss:.4f}' if eval_loss is not None else 'N/A'}")
    cost = metrics.get('estimated_cost_usd')
    if cost is None and billable:
        # Estimación: ml.g6.2xlarge cuesta ~$1.22/hr on-demand en us-west-2
        cost = round(billable / 3600 * 1.44, 2)
    print(f" Costo estimado:     ~${cost} USD")
    print("=" * 60)


In [ ]:
# Visualización rápida de duración, costo y loss
import matplotlib.pyplot as plt

if metrics:
    train_loss = metrics.get("final_train_loss")
    eval_loss = metrics.get("eval_loss")
    has_loss = train_loss is not None and eval_loss is not None

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    # Loss final train vs eval
    if has_loss:
        axes[0].bar(["train_loss", "eval_loss"],
                    [train_loss, eval_loss],
                    color=["#4C72B0", "#DD8452"])
        axes[0].set_title("Loss final (train vs eval)")
        axes[0].set_ylabel("loss")
        for i, v in enumerate([train_loss, eval_loss]):
            axes[0].text(i, v + 0.01, f"{v:.3f}", ha="center")
    else:
        axes[0].text(0.5, 0.5, "Métricas de loss\nno disponibles",
                     ha="center", va="center", transform=axes[0].transAxes, fontsize=12)
        axes[0].set_title("Loss final (no disponible)")

    # Entrenamiento puro vs wall-clock vs tiempo facturable
    train_min = (metrics.get("training_duration_seconds") or 0) / 60
    dur_min = (metrics.get("duration_seconds") or 0) / 60
    bill_min = (metrics.get("billable_time_seconds") or 0) / 60
    cost = metrics.get('estimated_cost_usd')
    if cost is None and bill_min > 0:
        cost = round(bill_min / 60 * 1.44, 2)
    labels = ["entrenamiento", "wall-clock", "facturable"]
    vals = [train_min, dur_min, bill_min]
    axes[1].bar(labels, vals, color=["#8172B3", "#55A868", "#C44E52"])
    axes[1].set_title(f"Tiempo del job (min) — costo ~${cost} USD")
    axes[1].set_ylabel("minutos")
    for i, v in enumerate(vals):
        if v > 0:
            axes[1].text(i, v + 0.3, f"{v:.1f}", ha="center")

    plt.tight_layout()
    plt.show()
else:
    print("No hay métricas para graficar.")


**Lectura de resultados.** Con aproximadamente 6.5K ejemplos y 3 épocas en una sola `ml.g6.2xlarge`, el bucle
de entrenamiento tomó **~1h21m** (`training_duration_seconds` ≈ 4853 s) y el *wall-clock*
facturable del job **~1h26m** (≈ 5170 s, incluye aprovisionamiento y descarga del modelo base),
con un costo estimado de **~$2 USD**. El `train_loss` y el `eval_loss` finales quedan muy
cercanos entre sí (≈0.96 vs ≈0.95), lo que indica que el adaptador aprendió el estilo y el
vocabulario del dominio **sin sobreajustarse**. Consistente con el objetivo del fine-tuning:
ajustar *formato y dominio*, no inyectar conocimiento exhaustivo (para eso se recomienda
complementar con RAG en producción).


### 8.2 Comparativo multi-modelo en paralelo (tiempo, memoria, loss)

Si se corrió `training/launch_training_job.py --model-keys` (Sección 7.4), existe un archivo
`training/parallel_run_<timestamp>.json` que referencia los `metrics_<job_name>.json`
individuales de cada modelo entrenado. Esta celda carga **todos** los `metrics_*.json`
disponibles (idealmente los de la corrida en paralelo más reciente) en un `DataFrame` de
`pandas`, para comparar entre modelos:

- Tiempo de entrenamiento (`training_duration_seconds`, wall-clock, facturable).
- Memoria pico de GPU (`gpu_peak_allocated_mb`, `gpu_peak_reserved_mb`) y memoria CPU
  (`cpu_rss_after_train_mb`).
- `final_train_loss` / `eval_loss` y tamaño del adaptador resultante (`adapter_size_mb`).
- Costo estimado y utilización de recursos (CloudWatch) por job.

> Si aún no se ha corrido el fine-tuning en paralelo, esta celda construye la tabla con los
> `metrics_*.json` que sí existan (por defecto, solo la corrida legacy de la Sección 8.1), y lo
> indica explícitamente, no falla.

In [ ]:
import pandas as pd

all_metric_files = sorted(glob.glob(str(TRAINING_DIR / "metrics_*.json")))
parallel_run_files = sorted(glob.glob(str(TRAINING_DIR / "parallel_run_*.json")))

rows = []
for m in all_metric_files:
    with open(m, encoding="utf-8") as f:
        d = json.load(f)
    resource = d.get("resource_utilization", {}) or {}
    rows.append({
        "job_name": d.get("training_job_name"),
        "model_key": d.get("model_key", d.get("training_job_name")),
        "base_model_id": d.get("base_model_id"),
        "instance_type": d.get("instance_type"),
        "status": d.get("training_job_status"),
        "duration_min": (d.get("duration_seconds") or 0) / 60,
        "training_loop_min": (d.get("training_duration_seconds") or 0) / 60,
        "billable_min": (d.get("billable_time_seconds") or 0) / 60,
        "train_samples_per_second": d.get("train_samples_per_second"),
        "final_train_loss": d.get("final_train_loss"),
        "eval_loss": d.get("eval_loss"),
        "gpu_peak_allocated_mb": d.get("gpu_peak_allocated_mb"),
        "gpu_peak_reserved_mb": d.get("gpu_peak_reserved_mb"),
        "cpu_rss_after_train_mb": d.get("cpu_rss_after_train_mb"),
        "adapter_size_mb": d.get("adapter_size_mb"),
        "trainable_params_pct": d.get("trainable_params_pct"),
        "estimated_cost_usd": d.get("estimated_cost_usd"),
        "gpu_utilization_avg_pct": resource.get("GPUUtilization_avg"),
        "memory_utilization_avg_pct": resource.get("MemoryUtilization_avg"),
    })

metrics_df = pd.DataFrame(rows)
print(f"Corridas de entrenamiento encontradas: {len(metrics_df)}")
print(f"Manifiestos de corridas en paralelo:   {[Path(p).name for p in parallel_run_files]}")
metrics_df

In [ ]:
import seaborn as sns

if len(metrics_df) >= 1:
    sns.set_theme(style="whitegrid")
    plot_df = metrics_df.dropna(subset=["model_key"]).copy()

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    # Tiempo de entrenamiento por modelo (wall-clock vs bucle puro)
    time_long = plot_df.melt(
        id_vars="model_key",
        value_vars=["duration_min", "training_loop_min"],
        var_name="tipo_de_tiempo",
        value_name="minutos",
    )
    sns.barplot(data=time_long, x="model_key", y="minutos", hue="tipo_de_tiempo", ax=axes[0, 0])
    axes[0, 0].set_title("Tiempo de entrenamiento por modelo")
    axes[0, 0].set_xlabel("")
    axes[0, 0].tick_params(axis="x", rotation=20)

    # Memoria pico de GPU por modelo
    mem_long = plot_df.melt(
        id_vars="model_key",
        value_vars=["gpu_peak_allocated_mb", "gpu_peak_reserved_mb"],
        var_name="tipo_de_memoria",
        value_name="MB",
    )
    sns.barplot(data=mem_long, x="model_key", y="MB", hue="tipo_de_memoria", ax=axes[0, 1])
    axes[0, 1].set_title("Memoria pico de GPU por modelo")
    axes[0, 1].set_xlabel("")
    axes[0, 1].tick_params(axis="x", rotation=20)

    # Loss final (train vs eval) por modelo
    loss_long = plot_df.melt(
        id_vars="model_key",
        value_vars=["final_train_loss", "eval_loss"],
        var_name="tipo_de_loss",
        value_name="loss",
    )
    sns.barplot(data=loss_long, x="model_key", y="loss", hue="tipo_de_loss", ax=axes[1, 0])
    axes[1, 0].set_title("Loss final por modelo (train vs eval)")
    axes[1, 0].set_xlabel("")
    axes[1, 0].tick_params(axis="x", rotation=20)
    

    # Costo estimado vs tamaño del adaptador
    sns.scatterplot(
        data=plot_df, x="adapter_size_mb", y="estimated_cost_usd",
        hue="model_key", s=140, ax=axes[1, 1],
    )
    axes[1, 1].set_title("Costo estimado vs tamaño del adaptador LoRA")
    axes[1, 1].set_xlabel("Tamaño del adaptador (MB)")
    axes[1, 1].set_ylabel("Costo estimado (USD)")

    plt.tight_layout()
    plt.show()
else:
    print("No hay corridas de entrenamiento para graficar todavía.")


### 8.3 Evaluación de rendimiento post-entrenamiento (calidad y velocidad)

Después de entrenar, `training/evaluate_models.py` (Sección 16 de
`docs/technical_documentation.md`, aplicada aquí a los adaptadores recién entrenados) mide,
para cada modelo:

- **Calidad**: perplexity sobre `eval.jsonl`.
- **Rendimiento**: tokens por segundo, tiempo al primer token (TTFT) y latencia total de
  generación (promedio, P50, P99), sobre un set fijo de prompts de benchmark.

Los resultados se guardan en `training/eval_metrics_<job_name>.json` por modelo. Esta celda
los carga con `pandas` y los visualiza con `seaborn` para comparar los modelos entre sí antes
de decidir cuál promover a la ruta de portabilidad (Sección 10).

> Requiere haber corrido `training/evaluate_models.py` (o `scripts/06b_evaluate_models.sh`)
> después del fine-tuning. Si aún no existen archivos `eval_metrics_*.json`, la celda lo
> indica y no falla.

In [ ]:
eval_metric_files = sorted(glob.glob(str(TRAINING_DIR / "eval_metrics_*.json")))

eval_rows = []
for m in eval_metric_files:
    with open(m, encoding="utf-8") as f:
        d = json.load(f)
    quality = d.get("quality", {}) or {}
    performance = d.get("performance", {}) or {}
    eval_rows.append({
        "job_name": d.get("job_name"),
        "model_key": d.get("model_key", d.get("job_name")),
        "base_model_id": d.get("base_model_id"),
        "perplexity": quality.get("perplexity"),
        "mean_eval_loss": quality.get("mean_eval_loss"),
        "tokens_per_second_avg": performance.get("tokens_per_second_avg"),
        "ttft_seconds_avg": performance.get("ttft_seconds_avg"),
        "ttft_seconds_p99": performance.get("ttft_seconds_p99"),
        "total_latency_seconds_avg": performance.get("total_latency_seconds_avg"),
        "total_latency_seconds_p99": performance.get("total_latency_seconds_p99"),
    })

eval_df = pd.DataFrame(eval_rows)
print(f"Archivos de evaluación encontrados: {len(eval_df)}")
eval_df

In [ ]:
if len(eval_df) >= 1:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    sns.barplot(data=eval_df, x="model_key", y="perplexity", ax=axes[0], color="#4C72B0")
    axes[0].set_title("Perplexity por modelo (menor es mejor)")
    axes[0].set_xlabel("")
    axes[0].tick_params(axis="x", rotation=20)

    sns.barplot(data=eval_df, x="model_key", y="tokens_per_second_avg", ax=axes[1], color="#55A868")
    axes[1].set_title("Throughput promedio (tokens/s)")
    axes[1].set_xlabel("")
    axes[1].tick_params(axis="x", rotation=20)

    latency_long = eval_df.melt(
        id_vars="model_key",
        value_vars=["ttft_seconds_avg", "total_latency_seconds_avg"],
        var_name="metrica_de_latencia",
        value_name="segundos",
    )
    sns.barplot(data=latency_long, x="model_key", y="segundos", hue="metrica_de_latencia", ax=axes[2])
    axes[2].set_title("Latencia promedio (TTFT vs total)")
    axes[2].set_xlabel("")
    axes[2].tick_params(axis="x", rotation=20)

    plt.tight_layout()
    plt.show()

    # Vista conjunta calidad vs velocidad: util para elegir el mejor
    # compromiso entre los dos ejes de la Sección 16 de
    # docs/technical_documentation.md.
    if len(eval_df) >= 2:
        g = sns.relplot(
            data=eval_df, x="tokens_per_second_avg", y="perplexity",
            hue="model_key", s=200, height=5, aspect=1.3,
        )
        g.set(title="Calidad (perplexity) vs velocidad (tokens/s) por modelo")
        plt.show()
else:
    print("No hay métricas de evaluación para graficar todavía. Corre training/evaluate_models.py primero.")

## 9. Despliegue de prueba en Bedrock AgentCore Runtime

El modelo se monta sobre **Bedrock AgentCore Runtime** (`us-west-2`) para pruebas end-to-end.

### 9.1 El agente: `agent_runtime/app.py`

Usa `bedrock_agentcore.BedrockAgentCoreApp`, que expone el contrato HTTP que AgentCore espera
(`/invocations` POST y `/ping` GET en el puerto 8080). Al arrancar:

1. Descarga el **adaptador LoRA** desde `s3://<bucket>/models/latest/` (solo el adaptador,
   decenas de MB; los ~3 GB del modelo base se bajan de Hugging Face Hub).
2. Carga `Qwen2.5-1.5B-Instruct` y le aplica el adaptador con `peft.PeftModel`.
3. Expone un `@app.entrypoint` que recibe `{"prompt": "..."}`, arma el chat con el mismo
   *system prompt* del dataset y devuelve `{"result": "..."}`.

### 9.2 La imagen: `agent_runtime/Dockerfile`

AgentCore Runtime requiere imágenes **`linux/arm64`**. El Dockerfile instala `torch` desde el
índice **CPU-only** de PyTorch (AgentCore no ofrece GPU), lo que además alinea el runtime con
el perfil de cómputo de `mx-central-1`.

### 9.3 El stack: `AgentRuntimeStack`

`agentcore.AgentRuntimeArtifact.from_asset(...)` hace que CDK construya la imagen ARM64 y la
publique en ECR; el construct `agentcore.Runtime` la despliega con las variables
`MODEL_BUCKET` / `MODEL_S3_PREFIX` y un rol con `s3:GetObject` sobre `models/*`.


In [ ]:
# Contrato de invocación del agente (esquema de payload)
ejemplo_payload = {"prompt": "Resume el objeto y los sujetos obligados de la Circular 3/2025 de Banxico."}
ejemplo_respuesta = {"result": "La Circular 3/2025 tiene por objeto... [respuesta del SLM]"}
print("Request :", json.dumps(ejemplo_payload, ensure_ascii=False, indent=2))
print("Response:", json.dumps(ejemplo_respuesta, ensure_ascii=False, indent=2))


> ⚠️ **ACCIÓN REQUERIDA — Ejecutar después del despliegue de `SlmAgentRuntimeStack`**
>
> Para invocar el agente desplegado, necesitas el ARN del AgentRuntime (output del stack).
> Puedes usar el código de abajo en una celda de Studio o en un script.


In [ ]:
# Invocación de prueba contra el runtime desplegado (requiere el ARN del
# AgentRuntime y credenciales). Ejecutar manualmente:
invoke_snippet = """
import boto3, json
client = boto3.client("bedrock-agentcore", region_name="us-west-2")
resp = client.invoke_agent_runtime(
    agentRuntimeArn="<AgentRuntimeArn del output de SlmAgentRuntimeStack>",
    payload=json.dumps({"prompt": "Lista las obligaciones de cumplimiento del fragmento..."}),
)
print(json.loads(resp["response"].read()))
"""
print(invoke_snippet)


> **Observación de latencia (documentada en `docs/portability_mx_central_1.md`):** en las
> pruebas, el runtime sirviendo el modelo **sin cuantizar** (bf16, `transformers` puro) sobre
> la CPU subyacente tomó entre **77 y 130 s** por invocación en frío (carga de modelo incluida).
> La cuantización a 4 bits (GGUF/OpenVINO) reduciría esto significativamente; ver sección 10.


## 10. Portabilidad a `mx-central-1` (Intel / ARM, sin GPU)

El requisito de negocio es poder desplegar en `mx-central-1`, pero esa región tiene dos
restricciones duras: **no hay GPU** y **AgentCore Runtime / Custom Model Import no están
disponibles**. Por eso el entrenamiento y la prueba se hacen en `us-west-2`, y el modelo se
**porta** después. La elección de Qwen2.5-1.5B + QLoRA fue precisamente lo que hace esto viable.

### Pipeline de conversión post-entrenamiento

<!-- TODO: Add diagram -->
<!-- Fuente editable: docs/diagrams/03_portabilidad_mx_central_1.drawio -->

1. **Fusionar** el adaptador con el modelo base (`merge_and_unload()`), obteniendo un modelo
   denso estándar.
2. **Ruta ARM/Graviton**: convertir a GGUF y cuantizar a `Q4_K_M` con `llama.cpp`; servir con
   `llama-server`/Ollama en EC2 `c7g`/`c8g`.
3. **Ruta Intel**: exportar a OpenVINO IR INT4 con `optimum-cli`; servir con OVMS en EC2 `c6in`.

Ambas rutas producen artefactos **solo-CPU**, aptos para `mx-central-1`. El dataset y el
fine-tuning **no se repiten**: el adaptador entrenado una vez se reutiliza para cualquier
región. El **contrato de la API** del agente se mantiene; solo cambia el runtime subyacente.


In [ ]:
# Esquema del merge del adaptador (paso 1 de la portabilidad).
merge_snippet = """
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", torch_dtype="float16")
merged = PeftModel.from_pretrained(base, "<adapter_dir>").merge_and_unload()
merged.save_pretrained("<merged_dir>")
AutoTokenizer.from_pretrained("<adapter_dir>").save_pretrained("<merged_dir>")

# Ruta ARM:   python llama.cpp/convert_hf_to_gguf.py <merged_dir> ... && llama-quantize ... Q4_K_M
# Ruta Intel: optimum-cli export openvino --model <merged_dir> --weight-format int4 <ov_dir>
"""
print(merge_snippet)


## Resumen y conclusiones

| Etapa | Herramienta | Resultado |
|---|---|---|
| Infraestructura | AWS CDK (3 stacks) | Bucket S3 + roles IAM en `us-west-2` |
| Adquisición | `scrape_cnbv.py`, `scrape_banxico.py` | PDFs de CNBV (9 sectores) y Banxico |
| Extracción | `extract_text.py` (pypdf) | ~778 documentos de texto limpio |
| Chunking | `chunk_documents.py` | ~15.5K chunks (~3K chars) |
| Dataset | `build_dataset.py` (Bedrock Claude Haiku 4.5) | ~7.2K ejemplos SFT (chat) |
| Catálogo de modelos | `model_catalog.py` (validado contra `technical_documentation.md`) | Qwen2.5-1.5B, Qwen3-0.6B |
| Fine-tuning single-job | SageMaker + `trl.SFTTrainer` | adaptador LoRA (Qwen2.5-1.5B), ~1h20m, ~$2 USD |
| Fine-tuning en paralelo | `launch_training_job.py --model-keys` (ThreadPoolExecutor) | N training jobs simultáneos, 1 por modelo del catálogo |
| Métricas de entrenamiento | `describe_training_job` + CloudWatch + `metrics.json` | duración, costo, loss, memoria GPU/CPU, utilización de recursos |
| Evaluación post-entrenamiento | `evaluate_models.py` (perplexity + tokens/s + latencia) | comparativo de calidad/rendimiento entre modelos, visualizado con `seaborn` |
| Despliegue | Bedrock AgentCore Runtime (ARM64) | agente de prueba end-to-end |
| Portabilidad | merge + GGUF / OpenVINO INT4 | listo para `mx-central-1` (solo CPU) |

**Cómo se cumple la especificación:**

- *Entrenamiento de un SLM en us-west-2 con SageMaker*: QLoRA de Qwen2.5-1.5B en `ml.g6.2xlarge` (flujo single-job original).
- *Fine-tuning con datos de cumplimiento CNBV/Banxico*: corpus scrapeado + dataset SFT del dominio.
- *Fine-tuning de varios modelos en jobs en paralelo*: `launch_training_job.py --model-keys` lanza y espera concurrentemente un job por modelo del catálogo (Sección 7.4).
- *Métricas del entrenamiento como tal (tiempo, memoria, etc.)*: duración (wall-clock/facturable/bucle puro), memoria pico de GPU (`torch.cuda.max_memory_*`) y CPU (`psutil`), throughput, loss, costo estimado y utilización de recursos vía CloudWatch, comparados entre modelos con `pandas`/`seaborn` (Sección 8.2).
- *Métricas de rendimiento de cada modelo resultante, evaluadas y visualizadas con seaborn*: `evaluate_models.py` mide perplexity, tokens/segundo, TTFT y latencia P50/P99 por modelo; la Sección 8.3 los visualiza con `seaborn` (barras comparativas y vista conjunta calidad-vs-velocidad).
- *Baja latencia y bajo costo*: modelos de 0.6B a 3.8B, adaptadores ligeros, costo de entrenamiento por job de pocos dólares.
- *Despliegue en AgentCore para prueba*: `agentcore.Runtime` con imagen ARM64.
- *Posibilidad de desplegar en mx-central-1 (solo Intel/ARM)*: ruta de cuantización GGUF/OpenVINO, validada contra las Secciones 4, 10 y 11 de `docs/technical_documentation.md`.

**Limitaciones conocidas** (de `docs/architecture.md`): el corpus es de tamaño modesto, ideal
para ajustar formato y vocabulario de dominio, no para inyectar conocimiento exhaustivo; para
eso se recomienda complementar con RAG. El scraping depende de la estructura HTML actual de los
sitios de origen. AgentCore Runtime aún no está disponible en `mx-central-1`, de ahí la
estrategia de portabilidad por cuantización. El entrenamiento en paralelo de varios modelos
multiplica el consumo de cuota de instancias GPU en `us-west-2` durante la ventana de
entrenamiento (Sección 7.4); y la evaluación de rendimiento (Sección 8.3) mide los adaptadores
sin cuantizar, como gate de calidad temprano; no sustituye el benchmark real post-cuantización
en `mx-central-1` descrito en la Sección 16 de `docs/technical_documentation.md`.